# Benchmark didáctico: PostgreSQL (SQL) vs MongoDB (NoSQL)

## Propósito
Este notebook ejecuta, paso a paso, un **workload equivalente** sobre dos motores:

- **PostgreSQL** (modelo relacional)
- **MongoDB** (documentos JSON)

Ambos gestionan el mismo escenario: **catálogo de productos** con operaciones frecuentes:

1. **Búsqueda** (search) de productos por texto.
2. **Actualización** (update) de stock.
3. **Inserción** (insert) de compras.

## Métricas evaluadas
Mediremos dos cuantitativas:

- **Latencia**: cuánto tarda cada operación (p50/p95).
- **Throughput**: cuántas operaciones por segundo (ops/s).

Y discutiremos dos cualitativas:

- **Complejidad de desarrollo**: modelado + consultas.
- **Operación y mantenimiento**: replicación, escalado, tooling.

## Cómo usar
1. Levanta el entorno Docker (Postgres + pgAdmin + MongoDB + mongo-express + Jupyter).
2. Abre este notebook en el navegador.
3. Ejecuta las celdas en orden.
4. Ajusta `OPS` y `CONCURRENCY` para explorar comportamiento.

> Nota: este benchmark es **educativo**. No sustituye herramientas como `pgbench` o `YCSB`.


## 1) Configuración de conexión

El contenedor de Jupyter recibe variables de entorno con las credenciales y hosts internos de Docker:

- Postgres: `postgres:5432`, DB `catalogo`
- Mongo: `mongodb:27017`, DB `catalogo`

En esta celda:
- leemos variables de entorno,
- inicializamos clientes,
- y hacemos un `ping` sencillo a cada motor.


In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
from pymongo import MongoClient
import psycopg

PG_HOST = os.getenv("PG_HOST", "postgres")
PG_PORT = int(os.getenv("PG_PORT", "5432"))
PG_DB   = os.getenv("PG_DB", "catalogo")
PG_USER = os.getenv("PG_USER", "demo")
PG_PASS = os.getenv("PG_PASS", "demo")

MONGO_URI = os.getenv("MONGO_URI", "mongodb://mongodb:27017")
MONGO_DB  = os.getenv("MONGO_DB", "catalogo")

# Parámetros del benchmark (puedes modificar)
OPS = int(os.getenv("OPS", "3000"))           # operaciones totales
CONCURRENCY = int(os.getenv("CONCURRENCY", "20"))  # hilos concurrentes

print("Postgres:", PG_HOST, PG_PORT, PG_DB, PG_USER)
print("Mongo:", MONGO_URI, MONGO_DB)
print("OPS:", OPS, "CONCURRENCY:", CONCURRENCY)

# Conexión Mongo (ping)
mongo_client = MongoClient(MONGO_URI)
mongo_db = mongo_client[MONGO_DB]
print("Mongo ping:", mongo_db.command("ping"))

# Conexión Postgres (ping)
conninfo = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASS}"
with psycopg.connect(conninfo, autocommit=True) as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT 1;")
        print("Postgres ping:", cur.fetchone())


## 2) Definición del workload (equivalente)

Ejecutamos el **mismo mix de operaciones** en ambos motores:

- **50% Search**
  - Postgres: `ILIKE '%Premium%'` o `ILIKE '%Base%'` sobre `productos.nombre`.
  - MongoDB: `$text: {$search: 'Premium'}` o `$search: 'Base'` sobre `productos.nombre`.
- **30% Update**
  - Disminuir stock de un producto aleatorio.
- **20% Insert**
  - Insertar una compra (línea).

### ¿Por qué este mix?
- Simula un patrón de e-commerce: muchas lecturas, menos escrituras.
- Es simple de explicar y permite ver diferencias de rendimiento.

En términos de arquitectura:
- Postgres resuelve integridad/relaciones a nivel del motor (FK, constraints).
- MongoDB prioriza flexibilidad; integridad suele gestionarse en la aplicación.


In [ ]:
def percentile(values, p):
    return float(np.percentile(values, p)) if values else 0.0

def summarize(label, lat_ms, t_start):
    dur = time.time() - t_start
    ops_done = len(lat_ms)
    thr = ops_done / dur if dur > 0 else 0.0
    return {
        "motor": label,
        "ops": ops_done,
        "dur_s": round(dur, 3),
        "throughput_ops_s": round(thr, 1),
        "p50_ms": round(percentile(lat_ms, 50), 3),
        "p95_ms": round(percentile(lat_ms, 95), 3),
        "max_ms": round(max(lat_ms) if lat_ms else 0.0, 3),
    }

def make_rng(seed):
    return random.Random(seed)


## 3) Benchmark en PostgreSQL

### Qué hace cada operación
- **Search**: consulta por texto parcial sobre `productos.nombre`.
- **Update**: decrementa stock.
- **Insert**: registra una compra (línea).

### Nota didáctica
Postgres ofrece consistencia fuerte (ACID) y un modelo relacional explícito.


In [ ]:
import threading

def pg_worker(seed, lat_ms):
    rng = make_rng(seed)
    conninfo = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASS}"
    with psycopg.connect(conninfo, autocommit=True) as conn:
        with conn.cursor() as cur:
            for _ in range(OPS // CONCURRENCY):
                op = rng.random()
                t0 = time.time()

                if op < 0.50:
                    term = "Premium" if rng.random() < 0.5 else "Base"
                    cur.execute(
                        "SELECT producto_id, nombre, precio, stock "
                        "FROM productos WHERE nombre ILIKE %s LIMIT 20;",
                        (f"%{term}%",)
                    )
                    cur.fetchall()

                elif op < 0.80:
                    pid = rng.randint(1, 10000)
                    cur.execute(
                        "UPDATE productos "
                        "SET stock = GREATEST(stock - 1, 0), actualizado_en = NOW() "
                        "WHERE producto_id = %s;",
                        (pid,)
                    )

                else:
                    cid = rng.randint(1, 2000)
                    pid = rng.randint(1, 10000)
                    qty = rng.randint(1, 4)
                    cur.execute("SELECT precio FROM productos WHERE producto_id=%s;", (pid,))
                    price = cur.fetchone()[0]
                    cur.execute(
                        "INSERT INTO compras(cliente_id, producto_id, cantidad, precio_unitario) "
                        "VALUES (%s,%s,%s,%s);",
                        (cid, pid, qty, price)
                    )

                lat_ms.append((time.time() - t0) * 1000.0)

def run_postgres():
    lat_ms = []
    t_start = time.time()

    threads = []
    for i in range(CONCURRENCY):
        t = threading.Thread(target=pg_worker, args=(1000 + i, lat_ms))
        t.start()
        threads.append(t)
    for t in threads:
        t.join()

    return summarize("PostgreSQL", lat_ms, t_start), lat_ms

pg_summary, pg_lat = run_postgres()
pg_summary


## 4) Benchmark en MongoDB

### Qué hace cada operación
- **Search**: índice de texto con `$text`.
- **Update**: decremento con `$inc`.
- **Insert**: inserción de documento compra.

MongoDB trabaja con documentos JSON; en esta comparación usamos colecciones separadas
para mantener equivalencia con el modelo relacional.


In [ ]:
def mongo_worker(seed, lat_ms, db):
    rng = make_rng(seed)
    productos = db.productos
    compras = db.compras

    for _ in range(OPS // CONCURRENCY):
        op = rng.random()
        t0 = time.time()

        if op < 0.50:
            term = "Premium" if rng.random() < 0.5 else "Base"
            list(productos.find(
                {"$text": {"$search": term}},
                {"nombre": 1, "precio": 1, "stock": 1}
            ).limit(20))

        elif op < 0.80:
            pid = rng.randint(1, 10000)
            productos.update_one(
                {"_id": pid},
                {"$inc": {"stock": -1}, "$set": {"actualizadoEn": time.time()}}
            )

        else:
            cid = rng.randint(1, 2000)
            pid = rng.randint(1, 10000)
            qty = rng.randint(1, 4)
            prod = productos.find_one({"_id": pid}, {"precio": 1})
            price = prod["precio"] if prod else 10
            compras.insert_one({
                "clienteId": cid,
                "productoId": pid,
                "cantidad": qty,
                "precioUnitario": price,
                "compradoEn": time.time()
            })

        lat_ms.append((time.time() - t0) * 1000.0)

def run_mongo():
    lat_ms = []
    t_start = time.time()

    threads = []
    for i in range(CONCURRENCY):
        t = threading.Thread(target=mongo_worker, args=(2000 + i, lat_ms, mongo_db))
        t.start()
        threads.append(t)
    for t in threads:
        t.join()

    return summarize("MongoDB", lat_ms, t_start), lat_ms

mongo_summary, mongo_lat = run_mongo()
mongo_summary


## 5) Microbenchmarks: Search vs Update vs Insert (por separado)

Hasta ahora ejecutamos un **workload mixto** (50/30/20). Eso es útil para una visión global, pero oculta *dónde* gana cada motor.

En esta sección ejecutamos **tres microbenchmarks**, manteniendo el mismo estilo de medición (p50/p95 y ops/s), pero aislando cada tipo de operación:

1. **Search-only** (solo búsquedas)
2. **Update-only** (solo actualizaciones de stock)
3. **Insert-only** (solo inserciones de compras)

### ¿Qué debes observar?
- En **Search**, suelen influir mucho los índices y el patrón de búsqueda (ILIKE vs $text).
- En **Update**, suelen influir locks/contention, journaling y patrón de escritura.
- En **Insert**, influyen el costo de escritura y el tamaño del documento/fila.

> Recomendación: ejecuta esta sección 2–3 veces y compara (hay variación por caché y carga del host).


In [ ]:
import threading

def run_microbench(label, worker_fn, *worker_args):
    lat_ms = []
    t_start = time.time()

    threads = []
    for i in range(CONCURRENCY):
        t = threading.Thread(target=worker_fn, args=(10000 + i, lat_ms, *worker_args))
        t.start()
        threads.append(t)
    for t in threads:
        t.join()

    return summarize(label, lat_ms, t_start), lat_ms

def compare_micro(pg_summ, mg_summ, op_name):
    rows = []
    for s in (pg_summ, mg_summ):
        rows.append({
            "operacion": op_name,
            "motor": s["motor"],
            "throughput_ops_s": s["throughput_ops_s"],
            "p50_ms": s["p50_ms"],
            "p95_ms": s["p95_ms"],
            "max_ms": s["max_ms"],
        })
    return rows


### 5.1 Microbenchmark: Search-only

In [ ]:
def pg_search_worker(seed, lat_ms):
    rng = make_rng(seed)
    conninfo = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASS}"
    with psycopg.connect(conninfo, autocommit=True) as conn:
        with conn.cursor() as cur:
            for _ in range(OPS // CONCURRENCY):
                term = "Premium" if rng.random() < 0.5 else "Base"
                t0 = time.time()
                cur.execute(
                    "SELECT producto_id, nombre, precio, stock "
                    "FROM productos WHERE nombre ILIKE %s LIMIT 20;",
                    (f"%{term}%",)
                )
                cur.fetchall()
                lat_ms.append((time.time() - t0) * 1000.0)

pg_search_summary, _ = run_microbench("PostgreSQL", pg_search_worker)
pg_search_summary


In [ ]:
def mongo_search_worker(seed, lat_ms, db):
    rng = make_rng(seed)
    productos = db.productos
    for _ in range(OPS // CONCURRENCY):
        term = "Premium" if rng.random() < 0.5 else "Base"
        t0 = time.time()
        list(productos.find(
            {"$text": {"$search": term}},
            {"nombre": 1, "precio": 1, "stock": 1}
        ).limit(20))
        lat_ms.append((time.time() - t0) * 1000.0)

mongo_search_summary, _ = run_microbench("MongoDB", mongo_search_worker, mongo_db)
mongo_search_summary


### 5.2 Microbenchmark: Update-only

In [ ]:
def pg_update_worker(seed, lat_ms):
    rng = make_rng(seed)
    conninfo = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASS}"
    with psycopg.connect(conninfo, autocommit=True) as conn:
        with conn.cursor() as cur:
            for _ in range(OPS // CONCURRENCY):
                pid = rng.randint(1, 10000)
                t0 = time.time()
                cur.execute(
                    "UPDATE productos "
                    "SET stock = GREATEST(stock - 1, 0), actualizado_en = NOW() "
                    "WHERE producto_id = %s;",
                    (pid,)
                )
                lat_ms.append((time.time() - t0) * 1000.0)

pg_update_summary, _ = run_microbench("PostgreSQL", pg_update_worker)
pg_update_summary


In [ ]:
def mongo_update_worker(seed, lat_ms, db):
    rng = make_rng(seed)
    productos = db.productos
    for _ in range(OPS // CONCURRENCY):
        pid = rng.randint(1, 10000)
        t0 = time.time()
        productos.update_one(
            {"_id": pid},
            {"$inc": {"stock": -1}, "$set": {"actualizadoEn": time.time()}}
        )
        lat_ms.append((time.time() - t0) * 1000.0)

mongo_update_summary, _ = run_microbench("MongoDB", mongo_update_worker, mongo_db)
mongo_update_summary


### 5.3 Microbenchmark: Insert-only

In [ ]:
def pg_insert_worker(seed, lat_ms):
    rng = make_rng(seed)
    conninfo = f"host={PG_HOST} port={PG_PORT} dbname={PG_DB} user={PG_USER} password={PG_PASS}"
    with psycopg.connect(conninfo, autocommit=True) as conn:
        with conn.cursor() as cur:
            for _ in range(OPS // CONCURRENCY):
                cid = rng.randint(1, 2000)
                pid = rng.randint(1, 10000)
                qty = rng.randint(1, 4)

                cur.execute("SELECT precio FROM productos WHERE producto_id=%s;", (pid,))
                price = cur.fetchone()[0]

                t0 = time.time()
                cur.execute(
                    "INSERT INTO compras(cliente_id, producto_id, cantidad, precio_unitario) "
                    "VALUES (%s,%s,%s,%s);",
                    (cid, pid, qty, price)
                )
                lat_ms.append((time.time() - t0) * 1000.0)

pg_insert_summary, _ = run_microbench("PostgreSQL", pg_insert_worker)
pg_insert_summary


In [ ]:
def mongo_insert_worker(seed, lat_ms, db):
    rng = make_rng(seed)
    productos = db.productos
    compras = db.compras

    for _ in range(OPS // CONCURRENCY):
        cid = rng.randint(1, 2000)
        pid = rng.randint(1, 10000)
        qty = rng.randint(1, 4)

        prod = productos.find_one({"_id": pid}, {"precio": 1})
        price = prod["precio"] if prod else 10

        t0 = time.time()
        compras.insert_one({
            "clienteId": cid,
            "productoId": pid,
            "cantidad": qty,
            "precioUnitario": price,
            "compradoEn": time.time()
        })
        lat_ms.append((time.time() - t0) * 1000.0)

mongo_insert_summary, _ = run_microbench("MongoDB", mongo_insert_worker, mongo_db)
mongo_insert_summary


### 5.4 Tabla comparativa: ¿dónde “gana” cada motor?

Interpretación sugerida:
- **Menor p95**: suele indicar comportamiento más estable bajo concurrencia.
- **Mayor throughput**: mayor capacidad de servir operaciones bajo el mismo workload.


In [ ]:
rows = []
rows += compare_micro(pg_search_summary, mongo_search_summary, "search")
rows += compare_micro(pg_update_summary, mongo_update_summary, "update")
rows += compare_micro(pg_insert_summary, mongo_insert_summary, "insert")

micro_df = pd.DataFrame(rows)
micro_df


In [ ]:
import matplotlib.pyplot as plt

pivot_thr = micro_df.pivot(index="operacion", columns="motor", values="throughput_ops_s")
pivot_thr.plot(kind="bar")
plt.ylabel("Throughput (ops/s)")
plt.title("Throughput por operación (microbench)")
plt.show()

pivot_p95 = micro_df.pivot(index="operacion", columns="motor", values="p95_ms")
pivot_p95.plot(kind="bar")
plt.ylabel("Latencia p95 (ms)")
plt.title("Latencia p95 por operación (microbench)")
plt.show()


## 6) Comparación y visualización

Comparamos métricas y visualizamos la distribución de latencias.


In [ ]:
import matplotlib.pyplot as plt

df = pd.DataFrame([pg_summary, mongo_summary])
df


In [ ]:
plt.figure()
plt.hist(pg_lat, bins=50, alpha=0.7, label="PostgreSQL")
plt.hist(mongo_lat, bins=50, alpha=0.7, label="MongoDB")
plt.xlabel("Latencia (ms)")
plt.ylabel("Frecuencia")
plt.legend()
plt.title("Distribución de latencias (workload mixto)")
plt.show()


## Apéndice — ¿Qué significan las latencias p50 y p95?

En este benchmark medimos **latencias p50 y p95** para entender no solo el promedio,
sino **cómo se comporta el sistema en condiciones normales y bajo carga**.

### Latencia
La **latencia** es el tiempo que tarda una operación individual en completarse
(búsqueda, actualización o inserción), normalmente medida en **milisegundos (ms)**.

Cada operación tiene su propia latencia.

---

### p50 (percentil 50)
El **p50** es la **mediana**.

- El **50 % de las operaciones** tardan **menos o igual** que este valor.
- Representa el **comportamiento típico** del sistema.

Ejemplo:
```
p50 = 3 ms
```
Significa que la mitad de las operaciones responden en 3 ms o menos.

---

### p95 (percentil 95)
El **p95** representa el **peor comportamiento habitual**.

- El **95 % de las operaciones** responden **por debajo** de este valor.
- Solo el **5 % más lento** queda fuera.

Ejemplo:
```
p95 = 20 ms
```
Significa que 95 de cada 100 operaciones responden en 20 ms o menos.

---

### ¿Por qué no usar solo el promedio?
El promedio puede ocultar problemas.

Ejemplo:
- 95 operaciones tardan 3 ms
- 5 operaciones tardan 200 ms

Promedio ≈ 13 ms  
Pero la mayoría de usuarios ve ~3 ms, y unos pocos sufren esperas largas.

Por eso, en sistemas reales se analizan **percentiles (p50, p95, p99)**.

---

### Cómo interpretar p50 y p95 en esta práctica

- **p50 bajo** → sistema rápido en condiciones normales.
- **p95 alto** → colas, locks, IO o contención bajo carga.
- **p50 y p95 cercanos** → comportamiento estable y predecible.

En la comparación SQL vs NoSQL:
- p50 suele ser similar.
- p95 suele mostrar mejor las diferencias de diseño interno y concurrencia.

> **Idea clave**: no solo importa ser rápido, sino ser **predecible bajo carga**.
